# 10 · Resumen de resultados 07 → 09 — RiskVision, Grupo 6

> **Este notebook solo muestra y comprueba.** No entrena, no registra y **no abre la prueba**. Los resultados se generan con `scripts/run_etapas_07_09.py` (lógica en `src/proyecto/models/final_run.py`) y quedan en `reports/`. Los notebooks 07, 08 y 09 conservan sus permisos apagados a propósito.

Estado del modelo: `riskvision_lightgbm_e` v1, alias `champion`, en **MLflow local** (sin conexión a Databricks en esta máquina).

In [1]:
import json, sys
from pathlib import Path
import joblib, pandas as pd

ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / 'src' / 'proyecto').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from proyecto.data.split import make_xy
from proyecto.inference.predict import InputValidationError, predict_with_scores
from proyecto.models import summary

results, registry = summary.load_results(ROOT)
print('Umbral de validación (congelado antes de la prueba):', results['settings']['LightGBM original']['threshold'])
print('Umbral de trabajo de la app:', results['operating_threshold'])

Umbral de validación (congelado antes de la prueba): 0.16666666666666666
Umbral de trabajo de la app: 0.2


## 1 · Métricas: validación cruzada, reserva interna y prueba (abierta una sola vez)

La prueba ya había sido expuesta (EDA y torneo): es una evaluación **complementaria**, no independiente.

In [2]:
summary.metrics_table(results)

roc_auc  avg_precision  \
Validación cruzada  LightGBM original             0.7872         0.5593   
                    LightGBM original + sigmoid   0.7872         0.5593   
                    Logística E                   0.7795         0.5477   
                    Logística E + sigmoid         0.7796         0.5476   
Reserva interna     LightGBM original             0.7865         0.5633   
                    LightGBM original + sigmoid   0.7865         0.5633   
                    Logística E                   0.7736         0.5516   
                    Logística E + sigmoid         0.7736         0.5516   
Prueba (umbral 1/6) LightGBM original             0.7816         0.5647   
                    Logística E                   0.7690         0.5320   

                                                  brier  precision  recall  \
Validación cruzada  LightGBM original            0.1332     0.3799  0.7597   
                    LightGBM original + sigmoid  0.1332     0.3707  0.7722   
                    Logística E                  0.1347     0.3978  0.7195   
                    Logística E + sigmoid        0.1347     0.3952  0.7265   
Reserva interna     LightGBM original            0.1331     0.3767  0.7528   
                    LightGBM original + sigmoid  0.1331     0.3736  0.7604   
                    Logística E                  0.1351     0.3982  0.7076   
                    Logística E + sigmoid        0.1351     0.3964  0.7091   
Prueba (umbral 1/6) LightGBM original            0.1343     0.3833  0.7423   
                    Logística E                  0.1374     0.3929  0.6888   

                                                 flagged_share  penalty  
Validación cruzada  LightGBM original                   0.4424   0.5402  
                    LightGBM original + sigmoid         0.4609   0.5420  
                    Logística E                         0.4001   0.5512  
                    Logística E + sigmoid               0.4067   0.5485  
Reserva interna     LightGBM original                   0.4420   0.5488  
                    LightGBM original + sigmoid         0.4502   0.5470  
                    Logística E                         0.3930   0.5598  
                    Logística E + sigmoid               0.3957   0.5605  
Prueba (umbral 1/6) LightGBM original                   0.4283   0.5492  
                    Logística E                         0.3877   0.5795

## 2 · Diferencia entre modelos en la prueba

Logística menos LightGBM, con intervalo bootstrap del 95 %; negativo favorece a LightGBM.

In [3]:
summary.paired_difference(results)

,difference,low,high,valid_draws
roc_auc,-0.012606,-0.01844,-0.00586,1000
penalty,0.030333,0.007663,0.050679,1000


## 3 · Variables que más pesan y segmentos

In [4]:
display(pd.DataFrame(results['importance']).round(4))
pd.DataFrame(results['segments'])[['variable', 'segment', 'n', 'roc_auc', 'recall_1', 'flagged_share']].round(3)

,feature,auc_decrease_mean,auc_decrease_std
0,PAY_0,0.0803,0.0032
1,LIMIT_BAL,0.0211,0.0033
2,BILL_AMT1,0.0149,0.0007
3,BILL_AMT2,0.0105,0.0019
4,PAY_AMT2,0.0047,0.0019
5,PAY_AMT3,0.0038,0.0013
6,PAY_4,0.0035,0.0003
7,BILL_AMT4,0.0033,0.0005
8,EDUCATION,0.0027,0.0006
9,PAY_2,0.0027,0.0010


,variable,segment,n,roc_auc,recall_1,flagged_share
0,SEX,1,2386,0.794,0.790,0.487
1,SEX,2,3614,0.780,0.724,0.413
2,AGE,"[18, 30)",1919,0.785,0.762,0.458
3,AGE,"[30, 45)",3020,0.799,0.752,0.414
4,AGE,"[45, 60)",1010,0.749,0.743,0.502
5,AGE,"[60, 101)",51,0.794,0.667,0.353


## 4 · Registro del modelo

In [5]:
pd.Series({k: registry[k] for k in ('modelo', 'version', 'alias', 'destino', 'experimento', 'estado', 'manifest_sha256')})

modelo                                         riskvision_lightgbm_e
version                                                            1
alias                                                       champion
destino                                                        local
experimento                                   riskvision-08-registro
estado                                           approved_definitive
manifest_sha256    54b91d67442b65cd2a61639ba1f9d53fddc7dfa2af51b6...
dtype: str

## 5 · Modelo exportado para la app: caso normal, límite e inválido

Usa `models/model.joblib` y `models/input_schema.json`, igual que la app (que NO consulta Databricks).

In [6]:
model = joblib.load(ROOT / 'models/model.joblib')
schema = json.loads((ROOT / 'models/input_schema.json').read_text(encoding='utf-8'))
X, _ = make_xy(joblib.load(ROOT / 'data/processed/train.joblib'))
probabilities = pd.Series(model.predict_proba(X)[:, 1], index=X.index)
print('Regla de decisión:', schema['decision_rule'], '| umbral:', schema['decision_threshold'])
for label, index in summary.demo_cases(probabilities, schema['decision_threshold']).items():
    out = predict_with_scores(X.loc[[index]], model, schema).iloc[0]
    print(f"{label:28s} P(incumple)={out['proba_1']:.3f}  decisión={out['decision']}")

Regla de decisión: marca incumple (1) si la probabilidad es >= decision_threshold; punto de trabajo del grupo elegido con validacion cruzada, no un costo medido del banco | umbral: 0.2
Normal (menor riesgo)        P(incumple)=0.017  decisión=0
Límite (justo en el umbral)  P(incumple)=0.200  decisión=1
Alto riesgo                  P(incumple)=0.888  decisión=1


In [7]:
invalid = summary.invalid_age_example(X.loc[[probabilities.idxmin()]])
try:
    predict_with_scores(invalid, model, schema)
except InputValidationError as error:
    print('Dato inválido rechazado ->', error)

Dato inválido rechazado -> AGE: debe estar entre 18 y 100


## Conclusión

- LightGBM + paquete E: **AUC 0,787 en validación cruzada y 0,782 en la prueba**, por encima de la logística y muy por encima del azar (0,50).
- La app decide con el umbral de trabajo 0,20 (34 % de clientes marcados en validación cruzada): es una **hipótesis del grupo**, no un costo medido.
- Pendientes: registro en Databricks (requiere `.env`), confirmar el costo real con el instructor y el despliegue público.